In [1]:
import asyncio
import json
import logging
import pandas as pd
from typing import List, Dict, Any, Optional, Set
import time
import nest_asyncio
from pathlib import Path
import tiktoken  # For token counting
from concurrent.futures import ThreadPoolExecutor, as_completed

# Enable nested event loops for Jupyter
nest_asyncio.apply()

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Import your existing conversation generator
import sys
sys.path.append('/home/sagemaker-user/csbai/multiturn_rl')

# Import reward calculation modules
from evaluation.metrics.hit_checker import calculate_hit_rate_batch
from evaluation.metrics.interactivity import evaluate_interactivity_batch_async_with_reasoning


class ConversationRewardEvaluator:
    """Evaluator for conversation rewards with multiple metrics"""
    
    def __init__(self, reward_config: Dict[str, Any]):
        self.config = reward_config
        self.weights = reward_config.get('weights', [1., 1., -0.01])
        self.interactivity_model = reward_config.get('interactivity_model', 
                                                    "us.anthropic.claude-3-7-sonnet-20250219-v1:0")
        self.max_workers = reward_config.get('max_workers', 3)
        self.max_turns = reward_config.get('max_turns', 10)
        self.max_tokens = reward_config.get('max_tokens', 512)
        self.encoding_name = reward_config.get('encoding_name', 'cl100k_base')
        logger.info(f"🏆 Reward Evaluator initialized: weights={self.weights}, max_tokens={self.max_tokens}")
    
    def load_conversation_pairs_csv(self, csv_path: str) -> pd.DataFrame:
        """Load conversation pairs from CSV"""
        logger.info(f"Loading conversation pairs from {csv_path}")
        df = pd.read_csv(csv_path)
        
        # Parse JSON strings
        json_columns = ['original_conversation', 'first_conversation', 'second_conversation']
        for col in json_columns:
            df[f'{col}_parsed'] = df[col].apply(self._safe_json_loads)
        
        # Parse existing rewards if they exist
        if 'chosen_reward' in df.columns:
            df['chosen_reward_parsed'] = df['chosen_reward'].apply(self._safe_json_loads)
            df['rejected_reward_parsed'] = df['rejected_reward'].apply(self._safe_json_loads)
        
        # Filter successful pairs
        successful_pairs = df[df['both_successful'] == True].copy()
        logger.info(f"Loaded {len(successful_pairs)}/{len(df)} successful pairs")
        return successful_pairs
    
    def _safe_json_loads(self, json_str):
        """Safely parse JSON string"""
        if pd.isna(json_str) or json_str is None:
            return None
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            return None

    def calculate_token_efficiency(self, conversation: List[Dict[str, str]], 
                             max_tokens: int = 512,
                             encoding_name: str = "cl100k_base") -> float:
        """Calculate token efficiency score (tokens used / max_tokens)"""
        try:
            encoding = tiktoken.get_encoding(encoding_name)
            total_tokens = 0
            
            for message in conversation:
                if message.get('role') == 'assistant':
                    content = message.get('content', '')
                    tokens = encoding.encode(content)
                    total_tokens += len(tokens)
            
            return total_tokens / max_tokens
        except Exception as e:
            logger.warning(f"Error counting tokens: {e}")
            return 0.0
    
    async def evaluate_metrics(self, conversations: List[List[Dict]], 
                              contexts: List[Dict], 
                              metrics: Set[str]) -> Dict[str, List[float]]:
        """Evaluate specified metrics for conversations"""
        results = {}
        
        if 'hit_rate' in metrics:
            logger.info("📊 Evaluating hit rates...")
            conversation_data = [{
                'ground_truth': ctx['ground_truth'],
                'generated_conversation': conv
            } for conv, ctx in zip(conversations, contexts)]
            
            hit_results = calculate_hit_rate_batch(conversation_data)
            results['hit_rate'] = [score for score, _ in hit_results]
        
        if 'interactivity' in metrics:
            logger.info("🎭 Evaluating interactivity...")
            scores, reasonings = await evaluate_interactivity_batch_async_with_reasoning(
                conversations, 
                model_id=self.interactivity_model,
                max_workers=self.max_workers,
                max_turns=self.max_turns
            )
            # Handle None values
            results['interactivity'] = [s if s is not None else 0.0 for s in scores]
            results['interactivity_reasoning'] = [r if r is not None else "" for r in reasonings]

        if 'token_efficiency' in metrics:
            logger.info("📝 Evaluating token efficiency...")
            results['token_efficiency'] = self._evaluate_token_efficiency_parallel(conversations)
        
        return results
    
    def _evaluate_token_efficiency_parallel(self, conversations: List[List[Dict]]) -> List[float]:
        """Evaluate token efficiency in parallel"""
        def calc_efficiency(conv):
            return self.calculate_token_efficiency(conv, self.max_tokens, self.encoding_name)
        
        scores = [0.0] * len(conversations)
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            future_to_idx = {executor.submit(calc_efficiency, conv): idx 
                           for idx, conv in enumerate(conversations)}
            
            for future in as_completed(future_to_idx):
                idx = future_to_idx[future]
                try:
                    scores[idx] = future.result()
                except Exception as e:
                    logger.warning(f"Token efficiency error for conv {idx}: {e}")
                    scores[idx] = 0.0
        
        return scores
    
    async def evaluate_conversation_pairs(self, df: pd.DataFrame, 
                                        metrics_to_evaluate: Set[str]) -> List[Dict]:
        """Main evaluation function"""
        logger.info(f"🏆 Evaluating {len(df)} pairs for metrics: {metrics_to_evaluate}")
        
        # Prepare conversations and contexts
        conversations, contexts, mapping = self._prepare_conversations_and_contexts(df)
        
        # Evaluate new metrics
        new_scores = await self.evaluate_metrics(conversations, contexts, metrics_to_evaluate)
        
        # Create results
        results = []
        for idx, row in df.iterrows():
            # Get existing rewards or create empty dicts
            existing_chosen = getattr(row, 'chosen_reward_parsed', None) or {}
            existing_rejected = getattr(row, 'rejected_reward_parsed', None) or {}
            
            # Find scores for this pair's conversations
            first_scores, second_scores, first_reasoning, second_reasoning = self._extract_scores_for_pair(
                idx, mapping, new_scores, existing_chosen, existing_rejected
            )
            
            # Calculate combined scores and determine chosen/rejected
            chosen_conv, rejected_conv, chosen_reward, rejected_reward, chosen_reasoning, rejected_reasoning = self._determine_chosen_rejected(
                row, first_scores, second_scores, first_reasoning, second_reasoning
            )
            
            result = {
                'id': row['id'],
                'ground_truth': row['ground_truth'],
                'original_conversation': row['original_conversation_parsed'],
                'chosen_conversation': chosen_conv,
                'rejected_conversation': rejected_conv,
                'chosen_reward': chosen_reward,
                'rejected_reward': rejected_reward,
                'chosen_reasoning': chosen_reasoning,
                'rejected_reasoning': rejected_reasoning,
                'reward_gap': chosen_reward.get('combined', 0) - rejected_reward.get('combined', 0),
                'metadata': self._create_metadata(metrics_to_evaluate)
            }
            results.append(result)
        
        self._log_results(results, new_scores)
        return results
    
    def _prepare_conversations_and_contexts(self, df: pd.DataFrame):
        """Prepare conversations and contexts for evaluation"""
        conversations = []
        contexts = []
        mapping = []  # (df_idx, 'first'/'second')
        
        for idx, row in df.iterrows():
            if row['first_conversation_parsed']:
                conversations.append(row['first_conversation_parsed'])
                contexts.append({'ground_truth': row['ground_truth']})
                mapping.append((idx, 'first'))
            
            if row['second_conversation_parsed']:
                conversations.append(row['second_conversation_parsed'])
                contexts.append({'ground_truth': row['ground_truth']})
                mapping.append((idx, 'second'))
        
        return conversations, contexts, mapping
    
    def _extract_scores_for_pair(self, pair_idx, mapping, new_scores, existing_chosen, existing_rejected):
        """Extract scores for a specific conversation pair"""
        first_scores = {}
        second_scores = {}
        first_reasoning = ""
        second_reasoning = ""
        
        # Add new metric scores and reasoning
        for conv_idx, (df_idx, conv_type) in enumerate(mapping):
            if df_idx == pair_idx:
                scores = {metric: scores_list[conv_idx] 
                         for metric, scores_list in new_scores.items()
                         if not metric.endswith('_reasoning')}
                
                # Extract reasoning if available
                reasoning = ""
                if 'interactivity_reasoning' in new_scores:
                    reasoning = new_scores['interactivity_reasoning'][conv_idx]
                
                if conv_type == 'first':
                    first_scores.update(scores)
                    first_reasoning = reasoning
                else:
                    second_scores.update(scores)
                    second_reasoning = reasoning
        
        # Try to preserve existing scores (simplified approach)
        if existing_chosen and existing_rejected:
            # Add any existing metrics that weren't re-evaluated
            for metric, value in existing_chosen.items():
                if metric not in first_scores and metric not in second_scores:
                    # This is an existing metric not being re-evaluated
                    # Simple heuristic: if first conversation has higher combined score,
                    # assume existing_chosen belongs to first
                    if 'combined' in existing_chosen and 'combined' in existing_rejected:
                        if existing_chosen['combined'] >= existing_rejected['combined']:
                            first_scores.setdefault(metric, existing_chosen[metric])
                            second_scores.setdefault(metric, existing_rejected[metric])
                        else:
                            first_scores.setdefault(metric, existing_rejected[metric])
                            second_scores.setdefault(metric, existing_chosen[metric])
        
        return first_scores, second_scores, first_reasoning, second_reasoning
    
    def _determine_chosen_rejected(self, row, first_scores, second_scores, first_reasoning, second_reasoning):
        """Determine which conversation is chosen based on scores"""
        # Calculate combined scores if we have the required metrics
        first_scores['combined'] = self._calculate_combined_score(first_scores)
        second_scores['combined'] = self._calculate_combined_score(second_scores)
        
        # Determine chosen/rejected based on combined score
        if first_scores['combined'] >= second_scores['combined']:
            return (row['first_conversation_parsed'], row['second_conversation_parsed'],
                   first_scores, second_scores, first_reasoning, second_reasoning)
        else:
            return (row['second_conversation_parsed'], row['first_conversation_parsed'],
                   second_scores, first_scores, second_reasoning, first_reasoning)
    
    def _calculate_combined_score(self, scores: Dict[str, float]) -> float:
        """Calculate combined score from available metrics"""
        available_metrics = ['hit_rate', 'interactivity', 'token_efficiency']
        present_metrics = [m for m in available_metrics if m in scores]
        
        if not present_metrics:
            return 0.0
        
        # If we have all three metrics, use configurable weights
        if len(present_metrics) == 3 and len(self.weights) == 3:
            return (self.weights[0] * scores['hit_rate'] + 
                   self.weights[1] * scores['interactivity'] + 
                   self.weights[2] * scores['token_efficiency'])
        
        # If we have hit_rate and interactivity (original case)
        elif 'hit_rate' in present_metrics and 'interactivity' in present_metrics:
            # Use first two weights, normalized
            total_weight = self.weights[0] + self.weights[1]
            if total_weight == 0:
                return 0.0
            w1 = self.weights[0] / total_weight
            w2 = self.weights[1] / total_weight
            return w1 * scores['hit_rate'] + w2 * scores['interactivity']
        
        # Fallback: equal weighting of available metrics
        else:
            return sum(scores[m] for m in present_metrics) / len(present_metrics)

    def _create_metadata(self, metrics_evaluated):
        """Create metadata dictionary"""
        return {
            'assistant': 'llama-3.2-1B-instruct',
            'user': 'claude-4-sonnet', 
            'weights': self.weights,
            'max_tokens': self.max_tokens,
            'evaluated_metrics': list(metrics_evaluated)
        }
    
    def _log_results(self, results, metric_scores):
        """Log evaluation statistics"""
        successful = sum(1 for r in results if r['chosen_reward'])
        logger.info(f"✅ Evaluated {successful}/{len(results)} pairs successfully")
    
    
    def save_results_to_csv(self, results: List[Dict], output_path: str):
        """Save results to CSV"""
        output_dir = Path(output_path).parent
        output_dir.mkdir(parents=True, exist_ok=True)
        
        csv_data = []
        for result in results:
            csv_row = {
                'id': result['id'],
                'ground_truth': result['ground_truth'],
                'original_conversation': json.dumps(result['original_conversation']),
                'chosen_conversation': json.dumps(result['chosen_conversation']) if result['chosen_conversation'] else None,
                'rejected_conversation': json.dumps(result['rejected_conversation']) if result['rejected_conversation'] else None,
                'chosen_reward': json.dumps(result['chosen_reward']) if result['chosen_reward'] else None,
                'rejected_reward': json.dumps(result['rejected_reward']) if result['rejected_reward'] else None,
                'reward_gap': result['reward_gap'],
                'metadata': json.dumps(result['metadata'])
            }
            csv_data.append(csv_row)
        
        df_output = pd.DataFrame(csv_data)
        df_output.to_csv(output_path, index=False)
        logger.info(f"Results saved to {output_path}")


async def main_evaluation(metrics_to_evaluate: Set[str] = None):
    """Main function to run reward evaluation"""
    if metrics_to_evaluate is None:
        metrics_to_evaluate = {'hit_rate', 'interactivity', 'token_efficiency'}
    
    logger.info(f"🎯 Evaluating metrics: {metrics_to_evaluate}")
    
    # Configuration
    dataset = "redial"
    alg = "vanilla"
    model = "llama3-2-1b-instruct"
    input_path = f"{dataset}/DPO_offline/{model}/conversation_pairs_generated.csv"
    output_path = f"{dataset}/DPO_offline/{model}/preference_pairs_with_rewards_final.csv"
    
    reward_config = {
        "weights": [1., 1., -0.01],
        "interactivity_model": "us.anthropic.claude-3-7-sonnet-20250219-v1:0", 
        "max_workers": 50,
        "max_turns": 11,
        "max_tokens": 512,
        "encoding_name": "cl100k_base"
    }
    
    # Run evaluation
    evaluator = ConversationRewardEvaluator(reward_config)
    df = evaluator.load_conversation_pairs_csv(input_path)
    # df = df[:10] # Used for testing
    start_time = time.time()
    results = await evaluator.evaluate_conversation_pairs(df, metrics_to_evaluate)
    logger.info(f"Evaluation completed in {time.time() - start_time:.2f}s")
    
    evaluator.save_results_to_csv(results, output_path)
    return results


# Example usage
print("🚀 Starting reward evaluation...")
evaluation_results = await main_evaluation({'hit_rate', 'interactivity', 'token_efficiency'})

INFO: 🎯 Evaluating metrics: {'hit_rate', 'token_efficiency', 'interactivity'}
INFO: 🏆 Reward Evaluator initialized: weights=[1.0, 1.0, -0.01], max_tokens=512
INFO: Loading conversation pairs from redial/DPO_offline/llama3-2-1b-instruct/conversation_pairs_generated.csv


🚀 Starting reward evaluation...


INFO: Loaded 8631/8631 successful pairs
INFO: 🏆 Evaluating 8631 pairs for metrics: {'hit_rate', 'token_efficiency', 'interactivity'}
INFO: 📊 Evaluating hit rates...
INFO: 🎭 Evaluating interactivity...


Completed 10/17262 interactivity evaluations
Completed 20/17262 interactivity evaluations
Completed 30/17262 interactivity evaluations
Completed 40/17262 interactivity evaluations
Completed 50/17262 interactivity evaluations
Completed 60/17262 interactivity evaluations
Completed 70/17262 interactivity evaluations
Completed 80/17262 interactivity evaluations
Completed 90/17262 interactivity evaluations
Completed 100/17262 interactivity evaluations
Completed 110/17262 interactivity evaluations
Completed 120/17262 interactivity evaluations
Completed 130/17262 interactivity evaluations
Completed 140/17262 interactivity evaluations
Completed 150/17262 interactivity evaluations
Completed 160/17262 interactivity evaluations
Completed 170/17262 interactivity evaluations
Completed 180/17262 interactivity evaluations
Completed 190/17262 interactivity evaluations
Completed 200/17262 interactivity evaluations
Completed 210/17262 interactivity evaluations
Completed 220/17262 interactivity evaluatio

INFO: 📝 Evaluating token efficiency...
INFO: ✅ Evaluated 8631/8631 pairs successfully
INFO: Evaluation completed in 2361.25s
INFO: Results saved to redial/DPO_offline/llama3-2-1b-instruct/preference_pairs_with_rewards_final.csv
